# 📊 Exploratory Data Analysis (EDA)

**Supply Chain Late Delivery Prediction**

---

## Overview

Comprehensive analysis of 180K+ supply chain orders to identify patterns, relationships, and predictors for late delivery risk.

### Analysis Sections

| Section | Focus |
|---------|-------|
| Data Quality | Missing values, data types |
| Delivery Status | Target variable distribution |
| Numeric Features | Sales, quantities, shipping days |
| Correlations | Feature relationships |
| Segments | Customer behavior patterns |
| Temporal | Time-based trends |
| Geographic | Regional patterns |
| Products | Category analysis |
| Shipping | Mode and carrier patterns |
| Financial | Revenue and profit analysis |

### Key Objectives

✅ Assess data quality and identify preprocessing needs  
✅ Discover patterns in delivery performance  
✅ Identify strong predictors for modeling  
✅ Generate feature engineering recommendations

---


In [6]:
# Setup & Data Loading
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.data.data_manager import load_raw

pd.set_option('display.max_columns', 50)

df = load_raw()
print(f"✅ Dataset: {df.shape[0]:,} orders × {df.shape[1]} features")


📂 Loading raw file: /Users/unclesam/Projects/supply-chain-ml-project/data/raw/DataCoSupplyChainDataset.csv
⚠️ UTF-8 decode failed. Retrying with Latin-1...
✅ Dataset: 180,519 orders × 53 features


## 1. Data Quality Assessment


In [7]:
# Missing Values Analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)

print("📋 Missing Values Summary:")
print(missing_df)

if len(missing_df) > 0:
    fig = px.bar(
        missing_df.reset_index(),
        x='Percent', y='index', orientation='h',
        title='<b>Missing Values by Column</b>',
        labels={'index': 'Column', 'Percent': 'Missing %'},
        color='Percent', color_continuous_scale='Reds'
    )
    fig.update_layout(height=350, yaxis={'categoryorder': 'total ascending'}, showlegend=False)
    fig.show()

print("\n💡 Only Product Description (100%) and Order Zipcode (86%) have significant missing values.")


📋 Missing Values Summary:
                      Count     Percent
Product Description  180519  100.000000
Order Zipcode        155679   86.239676
Customer Lname            8    0.004432
Customer Zipcode          3    0.001662



💡 Only Product Description (100%) and Order Zipcode (86%) have significant missing values.


## 2. Target Variable: Delivery Status


In [8]:
# Delivery Status Distribution
status_counts = df['Delivery Status'].value_counts()
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=('<b>Overall Distribution</b>', '<b>By Shipping Mode</b>')
)

# Donut chart
fig.add_trace(go.Pie(
    labels=status_counts.index, values=status_counts.values,
    hole=0.4, marker_colors=colors, textinfo='percent+label'
), row=1, col=1)

# Stacked bar by shipping mode
crosstab = pd.crosstab(df['Shipping Mode'], df['Delivery Status'], normalize='index') * 100
for status in crosstab.columns:
    fig.add_trace(go.Bar(
        name=status, x=crosstab.index, y=crosstab[status],
        text=[f'{v:.1f}%' for v in crosstab[status]], textposition='auto'
    ), row=1, col=2)

fig.update_layout(height=450, title_text='<b>Delivery Status Analysis</b>', barmode='stack')
fig.show()

late_pct = (df['Late_delivery_risk'].mean() * 100) if 'Late_delivery_risk' in df.columns else status_counts.get('Late delivery', 0) / len(df) * 100
print(f"\n🎯 Key Finding: {late_pct:.1f}% of deliveries are late → Classification target")



🎯 Key Finding: 54.8% of deliveries are late → Classification target


## 3. Numeric Feature Distributions


In [9]:
# Key Numeric Distributions
key_numeric = ['Sales', 'Order Item Quantity', 'Order Item Total', 'Days for shipping (real)']
existing = [c for c in key_numeric if c in df.columns]

fig = make_subplots(rows=2, cols=2, subplot_titles=[f'<b>{c}</b>' for c in existing[:4]])

for i, col in enumerate(existing[:4]):
    row, col_num = (i // 2) + 1, (i % 2) + 1
    data = df[col].dropna()

    fig.add_trace(go.Histogram(x=data, nbinsx=50, marker_color='#3498db', opacity=0.8, name=col), row=row, col=col_num)
    fig.add_vline(x=data.median(), line_dash="dash", line_color="#e74c3c",
                  annotation_text=f"Median: {data.median():.1f}", row=row, col=col_num)

fig.update_layout(height=600, title_text='<b>Numeric Feature Distributions</b>', showlegend=False)
fig.show()

print("\n💡 Sales & Order Item Total are highly correlated • Shipping days: 2-6 days typical")



💡 Sales & Order Item Total are highly correlated • Shipping days: 2-6 days typical


## 4. Feature Correlations


In [10]:
# Correlation Analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns[:15]
corr_matrix = df[numeric_cols].corr()

# Heatmap
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values, x=corr_matrix.columns, y=corr_matrix.columns,
    colorscale='RdBu', zmid=0, text=corr_matrix.round(2).values,
    texttemplate='%{text}', textfont={"size": 9}, colorbar=dict(title="Corr")
))
fig.update_layout(title='<b>Feature Correlation Matrix</b>', width=800, height=700, yaxis=dict(autorange="reversed"))
fig.show()

# Target correlations
if 'Late_delivery_risk' in df.columns:
    target_corr = df[numeric_cols].corrwith(df['Late_delivery_risk']).sort_values(key=abs, ascending=False)
    fig2 = px.bar(x=target_corr.head(10).values, y=target_corr.head(10).index, orientation='h',
                  title='<b>Top Predictors of Late Delivery</b>', labels={'x': 'Correlation', 'y': 'Feature'},
                  color=target_corr.head(10).values, color_continuous_scale='RdBu', color_continuous_midpoint=0)
    fig2.update_layout(height=400, yaxis={'categoryorder': 'total ascending'}, showlegend=False)
    fig2.show()

print("\n💡 Key Predictors: Days for shipment (scheduled) shows strong negative correlation with late delivery")



💡 Key Predictors: Days for shipment (scheduled) shows strong negative correlation with late delivery


## 5. Customer Segments


In [11]:
# Customer Segment Analysis
seg_counts = df['Customer Segment'].value_counts()
colors = ['#3498db', '#e74c3c', '#2ecc71']

fig = make_subplots(rows=1, cols=2, subplot_titles=('<b>Order Volume</b>', '<b>Late Delivery Rate</b>'))

fig.add_trace(go.Bar(x=seg_counts.index, y=seg_counts.values, marker_color=colors, showlegend=False), row=1, col=1)

if 'Late_delivery_risk' in df.columns:
    late_rate = df.groupby('Customer Segment')['Late_delivery_risk'].mean() * 100
    fig.add_trace(go.Bar(x=late_rate.index, y=late_rate.values, marker_color=colors, showlegend=False), row=1, col=2)
    fig.add_hline(y=df['Late_delivery_risk'].mean()*100, line_dash="dash", line_color="red",
                  annotation_text="Avg", row=1, col=2)

fig.update_layout(height=400, title_text='<b>Customer Segment Analysis</b>')
fig.update_yaxes(title_text="Orders", row=1, col=1)
fig.update_yaxes(title_text="Late Rate (%)", row=1, col=2)
fig.show()

print("\n💡 Consumer segment dominates volume • Late delivery rate consistent across segments")



💡 Consumer segment dominates volume • Late delivery rate consistent across segments


## 6. Temporal Trends


In [12]:
# Temporal Analysis
date_cols = [c for c in df.columns if 'date' in c.lower()]
if date_cols:
    df['order_date_parsed'] = pd.to_datetime(df[date_cols[0]], errors='coerce')
    df_dates = df[df['order_date_parsed'].notna()].copy()
    df_dates['year_month'] = df_dates['order_date_parsed'].dt.to_period('M')
    df_dates['day_of_week'] = df_dates['order_date_parsed'].dt.day_name()

    monthly_orders = df_dates.groupby('year_month').size()
    monthly_sales = df_dates.groupby('year_month')['Sales'].sum() if 'Sales' in df_dates.columns else None
    monthly_late = df_dates.groupby('year_month')['Late_delivery_risk'].mean() * 100 if 'Late_delivery_risk' in df_dates.columns else None
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_counts = df_dates['day_of_week'].value_counts().reindex(day_order, fill_value=0)

    fig = make_subplots(rows=2, cols=2, subplot_titles=(
        '<b>Orders Over Time</b>', '<b>Sales Over Time</b>',
        '<b>Late Delivery Rate</b>', '<b>Orders by Day of Week</b>'))

    fig.add_trace(go.Scatter(x=monthly_orders.index.astype(str), y=monthly_orders.values,
                              mode='lines+markers', line=dict(width=2, color='#3498db')), row=1, col=1)
    if monthly_sales is not None:
        fig.add_trace(go.Scatter(x=monthly_sales.index.astype(str), y=monthly_sales.values,
                                  mode='lines+markers', line=dict(width=2, color='#2ecc71')), row=1, col=2)
    if monthly_late is not None:
        fig.add_trace(go.Scatter(x=monthly_late.index.astype(str), y=monthly_late.values,
                                  mode='lines+markers', line=dict(width=2, color='#e74c3c')), row=2, col=1)
    fig.add_trace(go.Bar(x=day_counts.index, y=day_counts.values, marker_color='#3498db'), row=2, col=2)

    fig.update_layout(height=650, title_text='<b>Temporal Analysis</b>', showlegend=False)
    fig.show()

print("\n💡 Seasonal trends identified • Late delivery rate stable over time")



💡 Seasonal trends identified • Late delivery rate stable over time


## 7. Geographic Patterns


In [13]:
# Geographic patterns
geo_cols = [c for c in df.columns if any(x in c.lower() for x in ['region', 'country', 'state', 'city', 'market'])]
if geo_cols:
    # Matplotlib version
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    region_col = None
    country_col = None

    # Region analysis
    if 'Market' in df.columns or any('market' in c.lower() for c in df.columns):
        region_col = [c for c in df.columns if 'market' in c.lower()][0]
        region_counts = df[region_col].value_counts().head(10)
        axes[0, 0].barh(region_counts.index, region_counts.values, color='steelblue')
        axes[0, 0].set_xlabel('Number of Orders')
        axes[0, 0].set_title('Top 10 Markets by Order Volume', fontweight='bold')
        axes[0, 0].invert_yaxis()

    # Country analysis
    if 'Order Country' in df.columns or any('country' in c.lower() for c in df.columns):
        country_col = [c for c in df.columns if 'country' in c.lower()][0]
        country_counts = df[country_col].value_counts().head(10)
        axes[0, 1].barh(country_counts.index, country_counts.values, color='green')
        axes[0, 1].set_xlabel('Number of Orders')
        axes[0, 1].set_title('Top 10 Countries by Order Volume', fontweight='bold')
        axes[0, 1].invert_yaxis()

        # Late delivery by country
        if 'Late_delivery_risk' in df.columns:
            late_by_country = df.groupby(country_col)['Late_delivery_risk'].mean().sort_values(ascending=False).head(10) * 100
            axes[1, 0].barh(late_by_country.index, late_by_country.values, color='coral')
            axes[1, 0].set_xlabel('Late Delivery Rate (%)')
            axes[1, 0].set_title('Top 10 Countries by Late Delivery Rate', fontweight='bold')
            axes[1, 0].axvline(df['Late_delivery_risk'].mean()*100, color='red', linestyle='--', label='Overall Average')
            axes[1, 0].legend()
            axes[1, 0].invert_yaxis()

    # Sales by region
    if 'Sales' in df.columns and region_col:
        sales_by_region = df.groupby(region_col)['Sales'].sum().sort_values(ascending=False).head(10)
        axes[1, 1].barh(sales_by_region.index, sales_by_region.values, color='purple')
        axes[1, 1].set_xlabel('Total Sales ($)')
        axes[1, 1].set_title('Top 10 Markets by Sales Revenue', fontweight='bold')
        axes[1, 1].invert_yaxis()

    plt.tight_layout()
    plt.show()

    # Plotly interactive version
    fig_plotly = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Top 10 Markets by Order Volume', 'Top 10 Countries by Order Volume',
                       'Top 10 Countries by Late Delivery Rate', 'Top 10 Markets by Sales Revenue'),
        specs=[[{"type": "bar"}, {"type": "bar"}],
               [{"type": "bar"}, {"type": "bar"}]]
    )

    if region_col:
        region_counts = df[region_col].value_counts().head(10)
        fig_plotly.add_trace(
            go.Bar(
                x=region_counts.values,
                y=region_counts.index,
                orientation='h',
                name='Orders',
                marker_color='steelblue'
            ),
            row=1, col=1
        )

    if country_col:
        country_counts = df[country_col].value_counts().head(10)
        fig_plotly.add_trace(
            go.Bar(
                x=country_counts.values,
                y=country_counts.index,
                orientation='h',
                name='Orders',
                marker_color='green'
            ),
            row=1, col=2
        )

        if 'Late_delivery_risk' in df.columns:
            late_by_country = df.groupby(country_col)['Late_delivery_risk'].mean().sort_values(ascending=False).head(10) * 100
            fig_plotly.add_trace(
                go.Bar(
                    x=late_by_country.values,
                    y=late_by_country.index,
                    orientation='h',
                    name='Late Rate',
                    marker_color='coral'
                ),
                row=2, col=1
            )
            fig_plotly.add_vline(
                x=df['Late_delivery_risk'].mean()*100,
                line_dash="dash",
                line_color="red",
                annotation_text="Overall Average",
                row=2, col=1
            )

    if 'Sales' in df.columns and region_col:
        sales_by_region = df.groupby(region_col)['Sales'].sum().sort_values(ascending=False).head(10)
        fig_plotly.add_trace(
            go.Bar(
                x=sales_by_region.values,
                y=sales_by_region.index,
                orientation='h',
                name='Sales',
                marker_color='purple'
            ),
            row=2, col=2
        )

    fig_plotly.update_layout(
        height=800,
        title_text="Geographic Analysis (Interactive)",
        showlegend=False
    )
    fig_plotly.update_xaxes(title_text="Number of Orders", row=1, col=1)
    fig_plotly.update_xaxes(title_text="Number of Orders", row=1, col=2)
    fig_plotly.update_xaxes(title_text="Late Delivery Rate (%)", row=2, col=1)
    fig_plotly.update_xaxes(title_text="Total Sales ($)", row=2, col=2)
    fig_plotly.update_yaxes(autorange="reversed")
    fig_plotly.show()

    print("\n💡 INTERPRETATION:")
    print("   • Identify high-volume vs high-risk regions")
    print("   • Geographic patterns in delivery performance")
    print("   • Regional sales concentration")


NameError: name 'plt' is not defined

## 8. Product Categories


In [ ]:
# Product category analysis
product_cols = [c for c in df.columns if any(x in c.lower() for x in ['category', 'product', 'subcategory'])]
if product_cols:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Category distribution
    if 'Category Name' in df.columns or any('category' in c.lower() for c in df.columns):
        cat_col = [c for c in df.columns if 'category' in c.lower() and 'sub' not in c.lower()][0]
        cat_counts = df[cat_col].value_counts().head(10)
        axes[0, 0].barh(cat_counts.index, cat_counts.values, color='steelblue')
        axes[0, 0].set_xlabel('Number of Orders')
        axes[0, 0].set_title('Top 10 Product Categories by Order Volume', fontweight='bold')
        axes[0, 0].invert_yaxis()

        # Late delivery by category
        if 'Late_delivery_risk' in df.columns:
            late_by_cat = df.groupby(cat_col)['Late_delivery_risk'].mean().sort_values(ascending=False).head(10) * 100
            axes[0, 1].barh(late_by_cat.index, late_by_cat.values, color='coral')
            axes[0, 1].set_xlabel('Late Delivery Rate (%)')
            axes[0, 1].set_title('Top 10 Categories by Late Delivery Rate', fontweight='bold')
            axes[0, 1].axvline(df['Late_delivery_risk'].mean()*100, color='red', linestyle='--', label='Overall Average')
            axes[0, 1].legend()
            axes[0, 1].invert_yaxis()

    # Subcategory analysis
    if 'Sub-Category' in df.columns or any('subcategory' in c.lower() for c in df.columns):
        subcat_col = [c for c in df.columns if 'subcategory' in c.lower() or 'sub-category' in c.lower()][0]
        subcat_counts = df[subcat_col].value_counts().head(10)
        axes[1, 0].barh(subcat_counts.index, subcat_counts.values, color='green')
        axes[1, 0].set_xlabel('Number of Orders')
        axes[1, 0].set_title('Top 10 Subcategories by Order Volume', fontweight='bold')
        axes[1, 0].invert_yaxis()

    # Product popularity (if product name exists)
    if 'Product Name' in df.columns or any('product name' in c.lower() for c in df.columns):
        prod_col = [c for c in df.columns if 'product name' in c.lower()][0]
        prod_counts = df[prod_col].value_counts().head(10)
        axes[1, 1].barh(prod_counts.index, prod_counts.values, color='purple')
        axes[1, 1].set_xlabel('Number of Orders')
        axes[1, 1].set_title('Top 10 Products by Order Volume', fontweight='bold')
        axes[1, 1].invert_yaxis()

    plt.tight_layout()
    plt.show()

    print("\n💡 INTERPRETATION:")
    print("   • Identify high-risk product categories")
    print("   • Product popularity patterns")
    print("   • Category-specific delivery challenges")


## 9. Shipping Patterns


In [ ]:
# Shipping mode and carrier analysis
shipping_cols = [c for c in df.columns if any(x in c.lower() for x in ['shipping', 'carrier', 'mode'])]
if shipping_cols:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Shipping mode distribution
    if 'Shipping Mode' in df.columns:
        mode_counts = df['Shipping Mode'].value_counts()
        axes[0, 0].pie(mode_counts.values, labels=mode_counts.index, autopct='%1.1f%%', startangle=90)
        axes[0, 0].set_title('Shipping Mode Distribution', fontweight='bold')

        # Late delivery by shipping mode
        if 'Late_delivery_risk' in df.columns:
            late_by_mode = df.groupby('Shipping Mode')['Late_delivery_risk'].mean().sort_values(ascending=False) * 100
            axes[0, 1].bar(late_by_mode.index, late_by_mode.values, color='coral')
            axes[0, 1].axhline(df['Late_delivery_risk'].mean()*100, color='red', linestyle='--', label='Overall Average')
            axes[0, 1].set_xlabel('Shipping Mode')
            axes[0, 1].set_ylabel('Late Delivery Rate (%)')
            axes[0, 1].set_title('Late Delivery Rate by Shipping Mode', fontweight='bold')
            axes[0, 1].tick_params(axis='x', rotation=45)
            axes[0, 1].legend()

    # Days for shipping (scheduled) distribution
    if 'Days for shipment (scheduled)' in df.columns:
        days_col = 'Days for shipment (scheduled)'
        axes[1, 0].hist(df[days_col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
        axes[1, 0].axvline(df[days_col].median(), color='red', linestyle='--', linewidth=2,
                          label=f'Median: {df[days_col].median():.1f}')
        axes[1, 0].set_xlabel('Scheduled Shipping Days')
        axes[1, 0].set_ylabel('Frequency')
        axes[1, 0].set_title('Distribution of Scheduled Shipping Days', fontweight='bold')
        axes[1, 0].legend()

        # Correlation with late delivery
        if 'Late_delivery_risk' in df.columns:
            corr = df[[days_col, 'Late_delivery_risk']].corr().iloc[0, 1]
            axes[1, 0].text(0.7, 0.9, f'Corr with Late: {corr:.3f}',
                           transform=axes[1, 0].transAxes, fontsize=10,
                           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # Shipping mode vs sales
    if 'Sales' in df.columns and 'Shipping Mode' in df.columns:
        sales_by_mode = df.groupby('Shipping Mode')['Sales'].sum().sort_values(ascending=False)
        axes[1, 1].bar(sales_by_mode.index, sales_by_mode.values, color='green')
        axes[1, 1].set_xlabel('Shipping Mode')
        axes[1, 1].set_ylabel('Total Sales ($)')
        axes[1, 1].set_title('Total Sales by Shipping Mode', fontweight='bold')
        axes[1, 1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    print("\n💡 INTERPRETATION:")
    print("   • Shipping mode is a key predictor of delivery performance")
    print("   • Scheduled days strongly correlate with late delivery")
    print("   • Trade-offs between shipping speed and reliability")


## 10. Financial Analysis


In [ ]:
# Financial metrics analysis
financial_cols = [c for c in df.columns if any(x in c.lower() for x in ['sales', 'profit', 'discount', 'cost', 'revenue'])]
if financial_cols:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Sales distribution
    if 'Sales' in df.columns:
        # Log scale for better visualization
        sales_clean = df['Sales'].dropna()
        axes[0, 0].hist(np.log1p(sales_clean), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
        axes[0, 0].set_xlabel('Log(Sales + 1)')
        axes[0, 0].set_ylabel('Frequency')
        axes[0, 0].set_title('Sales Distribution (Log Scale)', fontweight='bold')
        axes[0, 0].axvline(np.log1p(sales_clean.median()), color='red', linestyle='--',
                         label=f'Median: ${sales_clean.median():.2f}')
        axes[0, 0].legend()

    # Profit analysis
    if 'Order Profit Per Order' in df.columns or any('profit' in c.lower() for c in df.columns):
        profit_col = [c for c in df.columns if 'profit' in c.lower()][0]
        profit_clean = df[profit_col].dropna()
        axes[0, 1].hist(profit_clean, bins=50, color='green', edgecolor='white', alpha=0.8)
        axes[0, 1].axvline(profit_clean.median(), color='red', linestyle='--',
                         label=f'Median: ${profit_clean.median():.2f}')
        axes[0, 1].axvline(0, color='black', linestyle='-', linewidth=1)
        axes[0, 1].set_xlabel('Profit per Order ($)')
        axes[0, 1].set_ylabel('Frequency')
        axes[0, 1].set_title('Profit Distribution', fontweight='bold')
        axes[0, 1].legend()

        # Negative profit orders
        negative_profit = (profit_clean < 0).sum()
        print(f"\n⚠️ Orders with negative profit: {negative_profit:,} ({negative_profit/len(profit_clean)*100:.1f}%)")

    # Discount analysis
    if 'Order Item Discount Rate' in df.columns or any('discount' in c.lower() for c in df.columns):
        discount_col = [c for c in df.columns if 'discount' in c.lower()][0]
        discount_clean = df[discount_col].dropna()
        axes[1, 0].hist(discount_clean, bins=30, color='orange', edgecolor='white', alpha=0.8)
        axes[1, 0].axvline(discount_clean.median(), color='red', linestyle='--',
                          label=f'Median: {discount_clean.median():.1%}')
        axes[1, 0].set_xlabel('Discount Rate')
        axes[1, 0].set_ylabel('Frequency')
        axes[1, 0].set_title('Discount Rate Distribution', fontweight='bold')
        axes[1, 0].legend()

        # Discount vs late delivery
        if 'Late_delivery_risk' in df.columns:
            discount_late = df.groupby(pd.cut(discount_clean, bins=5))['Late_delivery_risk'].mean() * 100
            axes[1, 1].bar(range(len(discount_late)), discount_late.values, color='purple')
            axes[1, 1].set_xlabel('Discount Rate Bins')
            axes[1, 1].set_ylabel('Late Delivery Rate (%)')
            axes[1, 1].set_title('Late Delivery Rate by Discount Level', fontweight='bold')
            axes[1, 1].set_xticklabels([f'{i:.1%}' for i in discount_late.index.mid])
            axes[1, 1].axhline(df['Late_delivery_risk'].mean()*100, color='red', linestyle='--', label='Overall Average')
            axes[1, 1].legend()

    plt.tight_layout()
    plt.show()

    print("\n💡 INTERPRETATION:")
    print("   • Sales and profit distributions show business patterns")
    print("   • Discount strategies may affect delivery performance")
    print("   • Identify high-value vs low-value orders")


## 11. Customer Behavior


In [ ]:
# Customer behavior patterns
customer_cols = [c for c in df.columns if any(x in c.lower() for x in ['customer', 'order'])]
if customer_cols:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Customer segment analysis (already done, but add more detail)
    if 'Customer Segment' in df.columns:
        seg_counts = df['Customer Segment'].value_counts()
        axes[0, 0].bar(seg_counts.index, seg_counts.values, color=['#3498db', '#e74c3c', '#2ecc71'])
        axes[0, 0].set_xlabel('Customer Segment')
        axes[0, 0].set_ylabel('Number of Orders')
        axes[0, 0].set_title('Orders by Customer Segment', fontweight='bold')

        # Average order value by segment
        if 'Sales' in df.columns:
            avg_order_value = df.groupby('Customer Segment')['Sales'].mean()
            axes[0, 1].bar(avg_order_value.index, avg_order_value.values, color=['#3498db', '#e74c3c', '#2ecc71'])
            axes[0, 1].set_xlabel('Customer Segment')
            axes[0, 1].set_ylabel('Average Order Value ($)')
            axes[0, 1].set_title('Average Order Value by Segment', fontweight='bold')

    # Order quantity distribution
    if 'Order Item Quantity' in df.columns:
        qty_dist = df['Order Item Quantity'].value_counts().head(10)
        axes[1, 0].bar(qty_dist.index.astype(str), qty_dist.values, color='steelblue')
        axes[1, 0].set_xlabel('Order Quantity')
        axes[1, 0].set_ylabel('Frequency')
        axes[1, 0].set_title('Top 10 Order Quantities', fontweight='bold')
        axes[1, 0].tick_params(axis='x', rotation=45)

    # Customer lifetime value (if we can calculate)
    if 'Sales' in df.columns and 'Customer ID' in df.columns or any('customer' in c.lower() and 'id' in c.lower() for c in df.columns):
        customer_id_col = [c for c in df.columns if 'customer' in c.lower() and 'id' in c.lower()][0]
        customer_lifetime = df.groupby(customer_id_col)['Sales'].sum().sort_values(ascending=False)
        top_customers = customer_lifetime.head(20)
        axes[1, 1].barh(range(len(top_customers)), top_customers.values, color='purple')
        axes[1, 1].set_yticks(range(len(top_customers)))
        axes[1, 1].set_yticklabels([f'Customer {i+1}' for i in range(len(top_customers))])
        axes[1, 1].set_xlabel('Total Sales ($)')
        axes[1, 1].set_title('Top 20 Customers by Lifetime Value', fontweight='bold')
        axes[1, 1].invert_yaxis()

    plt.tight_layout()
    plt.show()

    # Plotly version
    if 'Customer Segment' in df.columns and 'Sales' in df.columns:
        fig_plotly = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Orders by Customer Segment', 'Average Order Value by Segment',
                           'Top 10 Order Quantities', 'Top 20 Customers by Lifetime Value')
        )

        seg_counts = df['Customer Segment'].value_counts()
        fig_plotly.add_trace(
            go.Bar(x=seg_counts.index, y=seg_counts.values, name='Orders', marker_color='steelblue'),
            row=1, col=1
        )

        avg_order_value = df.groupby('Customer Segment')['Sales'].mean()
        fig_plotly.add_trace(
            go.Bar(x=avg_order_value.index, y=avg_order_value.values, name='Avg Order Value', marker_color='green'),
            row=1, col=2
        )

        if 'Order Item Quantity' in df.columns:
            qty_dist = df['Order Item Quantity'].value_counts().head(10)
            fig_plotly.add_trace(
                go.Bar(x=qty_dist.index.astype(str), y=qty_dist.values, name='Frequency', marker_color='orange'),
                row=2, col=1
            )

        if 'Customer ID' in df.columns or any('customer' in c.lower() and 'id' in c.lower() for c in df.columns):
            customer_id_col = [c for c in df.columns if 'customer' in c.lower() and 'id' in c.lower()][0]
            customer_lifetime = df.groupby(customer_id_col)['Sales'].sum().sort_values(ascending=False).head(20)
            fig_plotly.add_trace(
                go.Bar(x=customer_lifetime.values, y=[f'Customer {i+1}' for i in range(len(customer_lifetime))],
                      orientation='h', name='Lifetime Value', marker_color='purple'),
                row=2, col=2
            )

        fig_plotly.update_layout(height=800, title_text="Customer Behavior Analysis (Interactive)", showlegend=False)
        fig_plotly.update_xaxes(title_text="Number of Orders", row=1, col=1)
        fig_plotly.update_xaxes(title_text="Average Order Value ($)", row=1, col=2)
        fig_plotly.update_xaxes(title_text="Frequency", row=2, col=1)
        fig_plotly.update_xaxes(title_text="Total Sales ($)", row=2, col=2)
        fig_plotly.update_yaxes(autorange="reversed", row=2, col=2)
        fig_plotly.show()

    print("\n💡 INTERPRETATION:")
    print("   • Customer segments show different ordering patterns")
    print("   • High-value customers drive significant revenue")
    print("   • Order quantity patterns inform inventory planning")


## 2.12 Comprehensive EDA Summary

In [ ]:
# EDA Summary
print("=" * 70)
print("📊 EXPLORATORY DATA ANALYSIS SUMMARY")
print("=" * 70)

late_rate = df['Late_delivery_risk'].mean() * 100 if 'Late_delivery_risk' in df.columns else 55

summary = f"""
📋 DATASET OVERVIEW
   • {len(df):,} orders × {len(df.columns)} features
   • Late Delivery Rate: {late_rate:.1f}% (classification target)
   • Data Completeness: {(1 - df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100:.1f}%

🎯 TOP PREDICTORS FOR LATE DELIVERY
   1. Shipping Mode (strongest predictor)
   2. Scheduled Shipping Days (negative correlation)
   3. Customer Segment
   4. Product Category

💡 KEY INSIGHTS
   • Shipping method is the primary driver of delivery delays
   • Same Day shipping has highest on-time rate
   • Late delivery rate consistent across customer segments
   • Seasonal patterns exist but are not dominant

⚠️  DATA LEAKAGE WARNING
   Do NOT use these as features:
   • delivery_status (target in different form)
   • days_for_shipping_(real) (only known after delivery)

✅ READY FOR NEXT STEP: Data Preprocessing
"""
print(summary)
print("=" * 70)
print("➡️ Next: Run 02_data_preprocessing.ipynb")
